<a href="https://colab.research.google.com/github/murtazaarsh7/ml_model/blob/main/adaboost_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier

In [ ]:
class adaboost:
  def __init__(self,n_estimators=10):
    self.n_estimators=n_estimators
    self.models=[]
    self.alphas=[]

  def weights_update(self,df):
    df['weights']=1/df.shape[0]
  def calc_wrong(self,df):

    miscalssified=df["y"]!=df["y_pred"]
    wrong_total=df[miscalssified]["weights"].sum()
    return wrong_total
  def calculate_weight(self, error):
    return 0.5*np.log((1-error)/(error+0.000005))
  def create_new_df(self,df):
    indices=[]
    for i in range(df.shape[0]):
      a=np.random.random()
      for idx,row in df.iterrows():
        if a>row["normal_lower"] and a<=row["normal_upper"]:
          indices.append(idx)
          break
    return indices

  def update_row_weights(self,row,error):
    if row["y"]==row["y_pred"]:
      return row["weights"]*np.exp(-error)
    else :
      return row["weights"]*np.exp(error)
  def fit(self,x,y):
    original_num_features = x.shape[1] # Store the original number of features

    df=pd.DataFrame(x)
    df['y']=y
    self.weights_update(df)
    for i in range(self.n_estimators):
      dt=DecisionTreeClassifier(max_depth=1)
      dt.fit(x,y)
      df['y_pred']=dt.predict(x)
      alpha=self.calculate_weight(self.calc_wrong(df))
      df['updated_weights']=df.apply(self.update_row_weights,axis=1,args=(alpha,))
      df['normalized']=df["updated_weights"]/df['updated_weights'].sum()
      df["normal_upper"]=np.cumsum(df["normalized"])
      df["normal_lower"]=df["normal_upper"]-df["normalized"]
      index_values=self.create_new_df(df)
      new_df=df.iloc[index_values]

      # Update x and y for the next iteration using only the original features
      x=new_df.iloc[:, :original_num_features].values # Selects only the original feature columns
      y=new_df["y"].values # Selects the 'y' column

      self.models.append(dt)
      self.alphas.append(alpha)
  def predict(self,X):

    final_pred=np.zeros(X.shape[0])

    for alpha,model in zip(self.alphas,self.models):
      pred=model.predict(X)
      final_pred+=alpha*pred

    return np.sign(final_pred)

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

In [ ]:
data = load_breast_cancer()

X = data.data
y = data.target
y = np.where(y == 0, -1, 1)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
model=adaboost()
model.fit(X_train,y_train)
# 1. initialize weights
# 2. train decision stump
# 3. calculate error
# 4. calculate alpha
# 5. update weights
# 6. resample data
# 7. repeat n_estimators times


In [ ]:
predictions = model.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score

print("Accuracy:", accuracy_score(y_test, predictions))

Accuracy: 0.8947368421052632
